# T-S Hybrid Masked CLIP for Student Engagement Recognition (Google Colab)

Notebook này tự động:
1. **Tải (clone) mã nguồn từ GitHub repository** của bạn về máy ảo Google Colab (GPU T4).
2. **Kết nối Google Drive ở chế độ chỉ đọc (Read-Only)** tới thư mục dữ liệu [ViMELD_Dataset](https://drive.google.com/drive/folders/17s0ROqYsZwXH3IKdCTrI6KR8p1DwWEtD) (tuyệt đối không chỉnh sửa bất kỳ tệp dữ liệu nào trên Drive).
3. **Kiểm thử 4 Module & ma trận T-S Hybrid Attention Mask** $(1577 \times 1577)$ dựa trên nghiên cứu [Falcon Perception (arXiv:2603.27365)](https://arxiv.org/abs/2603.27365).
4. **Huấn luyện mô hình với LoRA (Q, V) & Mixed Precision (FP16)** trên GPU T4 và lưu trọng số PEFT ra thư mục cục bộ.

## Bước 1: Kiểm tra GPU T4 và Clone Repository từ GitHub
Thay `GITHUB_REPO_URL` bên dưới bằng đường dẫn GitHub repository của bạn sau khi bạn đã đẩy folder dự án lên GitHub.

In [ ]:
!nvidia-smi

import os
from pathlib import Path

# Thay YOUR_GITHUB_USERNAME và tên repo bằng đường dẫn GitHub thực tế của bạn:
GITHUB_REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/ts-hybrid-masked-clip-engagement.git"
REPO_DIR = Path("/content/ts-hybrid-masked-clip-engagement")

if not REPO_DIR.exists():
    print(f"Đang tải repository từ: {GITHUB_REPO_URL} ...")
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    print(f"Thư mục {REPO_DIR} đã tồn tại, đang cập nhật git pull...")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Thư mục làm việc hiện tại:", os.getcwd())

## Bước 2: Cài đặt thư viện phụ thuộc

In [ ]:
!pip install -q -r requirements.txt

## Bước 3: Kiểm thử kiến trúc 4 Module, Ma trận T-S Hybrid Mask (1577x1577) & LoRA
Chạy kiểm thử độc lập để xác nhận kích thước tensor qua cả 4 Module:
- **Module 1 (`InputEncodingModule`)**: $E_{task} \in \mathbb{R}^{B \times 1 \times 768}$, $E_{box} \in \mathbb{R}^{B \times 8 \times 768}$, $E_{patch} \in \mathbb{R}^{B \times 8 \times 196 \times 768}$.
- **Module 2 (`SpatiotemporalInjectionModule`)**: Chuỗi nối $\text{Sequence} = [E_{task}] \oplus E_{box} \oplus E_{patch} \in \mathbb{R}^{B \times 1577 \times 768}$.
- **Module 3 (`TSHybridMaskedTransformer`)**: Ma trận mặt nạ $(1577, 1577)$ tuân thủ quy tắc XAI và chỉ mở khóa gradient cho LoRA ($Q, V$).
- **Module 4 (`EngagementHead`)**: Phân loại từ vector đầu ra của Task Token ở lớp cuối cùng với `CrossEntropyLoss`.

In [ ]:
!python train.py --verify_only

## Bước 4: Trực quan hóa Ma trận T-S Hybrid Attention Mask (XAI)

In [ ]:
import matplotlib.pyplot as plt
import torch
from src.model import build_ts_hybrid_attention_mask

# Tạo mask đầy đủ (T=8, N=196 -> L=1577) và mask thu nhỏ (T=4, N=8 -> L=37) để dễ quan sát từng ô
full_mask = build_ts_hybrid_attention_mask(num_frames=8, num_patches=196)
demo_mask = build_ts_hybrid_attention_mask(num_frames=4, num_patches=8)

print(f"Kích thước ma trận Hybrid Mask thực tế (T=8, N=196): {tuple(full_mask.shape)}")
print(f"Tỷ lệ kết nối được phép nhìn (Sparsity): {100.0 * full_mask.float().mean().item():.2f}% (tiết kiệm >87% chi phí Attention so với Full Attention!)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(demo_mask.numpy(), cmap="Blues", interpolation="nearest")
axes[0].set_title("Cấu trúc T-S Hybrid Mask minh họa (T=4, N=8 -> L=37)")
axes[0].set_xlabel("Key Token Index (Ai được nhìn)")
axes[0].set_ylabel("Query Token Index (Token đang xét)")

axes[1].imshow(full_mask.numpy(), cmap="Blues", interpolation="nearest")
axes[1].set_title("Ma trận T-S Hybrid Mask thực tế (T=8, N=196 -> L=1577)")
axes[1].set_xlabel("Key Token Index (0: Task, 1..8: Box, 9..1576: Patches)")
axes[1].set_ylabel("Query Token Index")
plt.tight_layout()
plt.show()

## Bước 5: Mount Google Drive và Đọc dữ liệu `ViMELD_Dataset` (Chế độ Read-Only tuyệt đối)
- Bộ nạp dữ liệu `ViMELDEngagementDataset` chỉ mở các tệp ở chế độ đọc (`"r"` / `mmap_mode="r"`).
- Toàn bộ tệp checkpoint khi huấn luyện được lưu vào `/content/checkpoints_ts_hybrid_clip` trên máy ảo Colab để **không bao giờ ghi hay chỉnh sửa vào thư mục dữ liệu trên Google Drive của bạn**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Đường dẫn tới thư mục ViMELD_Dataset trên Google Drive của bạn
# Nếu thư mục được chia sẻ trong 'Shared with me', bạn có thể tạo lối tắt (Add shortcut to Drive) vào MyDrive
DATA_ROOT = "/content/drive/MyDrive/ViMELD_Dataset"

# Tùy chọn: nếu bạn dùng bảng manifest từ Restructured_Dataset_NPY_Engagement:
MANIFEST_PATH = "/content/drive/MyDrive/Restructured_Dataset_NPY_Engagement/manifest.parquet"
if not Path(MANIFEST_PATH).exists():
    MANIFEST_PATH = None

print("Kiểm tra thư mục DATA_ROOT tồn tại:", Path(DATA_ROOT).exists())
print("Sử dụng MANIFEST_PATH:", MANIFEST_PATH)

## Bước 6: Khởi tạo Mô hình với Trọng số Pretrained `openai/clip-vit-base-patch16` & Huấn luyện trên GPU T4

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.dataset import ViMELDEngagementDataset
from src.model import StudentEngagementTSHybridCLIP
from src.trainer import configure_optimizer, evaluate, save_peft_checkpoint, train_epoch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Đang sử dụng thiết bị:", device)

# 1. Tải trọng số CLIP ViT-B/16 từ Hugging Face và bơm LoRA vào Q, V
model = StudentEngagementTSHybridCLIP(
    clip_model_name="openai/clip-vit-base-patch16",
    num_frames=8,          # T = 8 frames -> L = 1 + 8 + 1568 = 1577 tokens
    image_size=224,        # H = W = 224, patch = 16x16 -> N = 196 patches/frame
    embed_dim=768,         # Không gian nhúng chung D = 768
    num_classes=3,         # 0: Low, 1: Medium, 2: High Engagement
    lora_r=8,
    lora_alpha=16.0,
    load_pretrained_clip=True,
).to(device)

model.print_parameter_summary()

In [ ]:
# 2. Khởi tạo DataLoader (Read-Only) và chạy vòng lặp huấn luyện
if Path(DATA_ROOT).exists() or (MANIFEST_PATH is not None and Path(MANIFEST_PATH).exists()):
    train_ds = ViMELDEngagementDataset(
        root_dir=DATA_ROOT,
        split="train",
        num_frames=8,
        num_classes=3,
        manifest_path=MANIFEST_PATH,
        use_full_scene_video=True,  # Đặt False nếu chỉ muốn đọc nhanh từ 16 frame PNG trong seg_*
    )
    valid_ds = ViMELDEngagementDataset(
        root_dir=DATA_ROOT,
        split="valid",
        num_frames=8,
        num_classes=3,
        manifest_path=MANIFEST_PATH,
        use_full_scene_video=True,
    )
    print(f"Số lượng mẫu đọc được -> Train: {len(train_ds):,} | Valid: {len(valid_ds):,}")

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

    optimizer = configure_optimizer(model, lr_lora=2e-4, lr_new_modules=5e-4)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    EPOCHS = 5
    best_acc = -1.0
    ckpt_dir = Path("/content/checkpoints_ts_hybrid_clip")

    for epoch in range(1, EPOCHS + 1):
        print(f"\n=== Epoch {epoch}/{EPOCHS} ===")
        tr_metrics = train_epoch(model, train_loader, optimizer, criterion, device, scaler=scaler)
        print(f"Train Loss: {tr_metrics['loss']:.4f} | Train Acc: {tr_metrics['accuracy']:.2f}%")

        if len(valid_ds) > 0:
            va_metrics = evaluate(model, valid_loader, criterion, device)
            print(f"Valid Loss: {va_metrics['loss']:.4f} | Valid Acc: {va_metrics['accuracy']:.2f}%")
            if va_metrics["accuracy"] > best_acc:
                best_acc = va_metrics["accuracy"]
                save_peft_checkpoint(model, ckpt_dir / "best_peft_weights.pt", epoch, va_metrics)
                print("-> Đã lưu trọng số PEFT tốt nhất vào /content/checkpoints_ts_hybrid_clip/best_peft_weights.pt")
else:
    print("Chưa tìm thấy đường dẫn DATA_ROOT trên Drive. Hãy kiểm tra lại đường dẫn shortcut trong MyDrive!")